# Laboratorio COCO — Detección y Segmentación con YOLO (Transfer Learning)

**Talento Altamente Especializado — Inteligencia Artificial 2026**
**COCYTEN-Nayarit**

**Instructor:** M.Sc. Mario Iván López Valdovinos  
**Alumno:** Alejandro Campos Martínez  
**Fecha:** 24/07/2026  

---

## Introducción

Este proyecto implementa y evalúa dos modelos de red neuronal (YOLOv8) mediante **transfer learning** a partir de pesos preentrenados en COCO, para las tareas de:

1. **Detección de objetos** (bounding boxes)
2. **Segmentación de instancias** (máscaras a nivel de píxel)

Se evalúa el desempeño con las métricas estándar: Precision, Recall, F1-score, IoU, Dice y mAP (en los umbrales 0.50 y 0.50:0.95).

**Dataset:** se utiliza `coco128-seg`, un subconjunto oficial de 128 imágenes tomadas de COCO train2017, que ya incluye anotaciones de segmentación (polígonos) además de bounding boxes. Este subconjunto es distribuido por Ultralytics específicamente para validar pipelines de entrenamiento/transfer learning sin necesitar los ~20 GB del dataset COCO completo.

> Nota metodológica: dado que el ejercicio pide explícitamente *transfer learning* (no entrenamiento desde cero), partimos siempre de los pesos `yolov8s.pt` / `yolov8s-seg.pt` preentrenados en el COCO completo (80 clases), y hacemos fine-tuning sobre nuestra partición.


## 1. Instalación de librerías

Instalamos PyTorch (con soporte CUDA), Ultralytics YOLO, y las utilerías necesarias para manejo de datos, métricas y visualización.

In [1]:
# Si ya tienes PyTorch con CUDA instalado (recomendado, para no reinstalar torch),
# comenta la línea de torch y deja solo el resto.
%pip install ultralytics matplotlib pycocotools scikit-learn seaborn opencv-python --quiet

import ultralytics
ultralytics.checks()  # imprime versión, disponibilidad de CUDA, GPU detectada, etc.


Ultralytics 8.4.106 🚀 Python-3.11.15 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
Setup complete ✅ (16 CPUs, 15.3 GB RAM, 76.9/467.3 GB disk)


In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = 0 if torch.cuda.is_available() else "cpu"


PyTorch version: 2.13.0+cu130
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM total: 8.2 GB


## 2. Descarga del dataset COCO (subconjunto)

Usamos `coco128-seg.zip`, distribuido oficialmente por Ultralytics. Ya viene en formato YOLO (imágenes + labels `.txt` con polígonos normalizados), lo cual nos ahorra el paso de convertir las anotaciones originales de COCO (formato JSON estilo COCO API) a formato YOLO.

Estructura que se descarga:
```
coco128-seg/
├── images/train2017/   (128 imágenes)
└── labels/train2017/   (128 archivos .txt, uno por imagen, con polígonos de segmentación)
```

Como el dataset original solo trae una partición (`train2017`), en la sección 3 lo re-particionamos nosotros mismos en 70/15/15 como pide el laboratorio.


In [3]:
import os
from pathlib import Path
from ultralytics.utils.downloads import download

DATA_ROOT = Path("./dataset_coco")
DATA_ROOT.mkdir(exist_ok=True)

# Descarga oficial del subconjunto (imágenes + labels de segmentación en formato YOLO)
url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128-seg.zip"
download(url, dir=DATA_ROOT, unzip=True, delete=True)

IMAGES_DIR = DATA_ROOT / "coco128-seg" / "images" / "train2017"
LABELS_DIR = DATA_ROOT / "coco128-seg" / "labels" / "train2017"

print(f"Imágenes descargadas: {len(list(IMAGES_DIR.glob('*.jpg')))}")
print(f"Labels descargados:   {len(list(LABELS_DIR.glob('*.txt')))}")


WARNING ⚠️ Skipping dataset_coco/coco128-seg.zip unzip as destination directory /home/xandro/Documentos/Certificacion/TAE_IA_Modulo_4/Segmentación_y_detección/dataset_coco/coco128-seg is not empty.
Imágenes descargadas: 128
Labels descargados:   128


In [4]:
# Nombres de las 80 clases de COCO (mismo orden que usan los pesos preentrenados de YOLO)
COCO_CLASSES = ultralytics.YOLO("yolov8s.pt").names  # dict {id: nombre}
print(f"Número de clases COCO: {len(COCO_CLASSES)}")
list(COCO_CLASSES.items())[:10]


Número de clases COCO: 80


[(0, 'person'),
 (1, 'bicycle'),
 (2, 'car'),
 (3, 'motorcycle'),
 (4, 'airplane'),
 (5, 'bus'),
 (6, 'train'),
 (7, 'truck'),
 (8, 'boat'),
 (9, 'traffic light')]

## 3. Partición del dataset (70% train / 15% val / 15% test)

Reorganizamos las 128 imágenes descargadas en tres particiones independientes, manteniendo la correspondencia imagen–etiqueta. Usamos una semilla fija para que la partición sea reproducible.


In [5]:
import random
import shutil

random.seed(42)

SPLIT_ROOT = Path("./dataset_coco/split")
if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

all_images = sorted(IMAGES_DIR.glob("*.jpg"))
random.shuffle(all_images)

n_total = len(all_images)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
# el resto (15% o el residuo por redondeo) va a test
splits = {
    "train": all_images[:n_train],
    "val": all_images[n_train:n_train + n_val],
    "test": all_images[n_train + n_val:],
}

for split_name, img_list in splits.items():
    img_out = SPLIT_ROOT / "images" / split_name
    lbl_out = SPLIT_ROOT / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img_path in img_list:
        label_path = LABELS_DIR / (img_path.stem + ".txt")
        shutil.copy(img_path, img_out / img_path.name)
        if label_path.exists():
            shutil.copy(label_path, lbl_out / label_path.name)

print(f"Total de imágenes: {n_total}")
for split_name, img_list in splits.items():
    pct = len(img_list) / n_total * 100
    print(f"  {split_name:5s}: {len(img_list):4d} imágenes ({pct:.1f}%)")


Total de imágenes: 128
  train:   89 imágenes (69.5%)
  val  :   19 imágenes (14.8%)
  test :   20 imágenes (15.6%)


## 4. Preparación de las anotaciones

Ultralytics YOLO usa un único formato de archivo de texto por imagen, pero el **contenido** de cada línea cambia según la tarea:

**Detección** (bounding box) — cada línea:
```
<clase> <x_centro> <y_centro> <ancho> <alto>
```
Todas las coordenadas normalizadas entre 0 y 1 respecto al tamaño de la imagen.

**Segmentación** (máscara) — cada línea:
```
<clase> <x1> <y1> <x2> <y2> ... <xn> <yn>
```
Es decir, el polígono completo del contorno del objeto, también normalizado.

`coco128-seg` ya trae las anotaciones en formato de **polígono** (segmentación). Para la tarea de **detección** derivamos el bounding box de cada objeto calculando el rectángulo delimitador (mínimo y máximo en x, y) de su polígono — así generamos las etiquetas de detección a partir de las de segmentación, garantizando que ambas tareas usen exactamente los mismos objetos.


In [6]:
def polygon_to_bbox(coords):
    """Convierte una lista plana [x1,y1,x2,y2,...] normalizada a un bbox
    YOLO [x_centro, y_centro, ancho, alto], también normalizado."""
    xs = coords[0::2]
    ys = coords[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return x_center, y_center, width, height


def build_detection_labels(seg_labels_dir: Path, det_labels_dir: Path):
    det_labels_dir.mkdir(parents=True, exist_ok=True)
    for seg_file in seg_labels_dir.glob("*.txt"):
        lines_out = []
        with open(seg_file) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls = parts[0]
                coords = list(map(float, parts[1:]))
                xc, yc, w, h = polygon_to_bbox(coords)
                lines_out.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        with open(det_labels_dir / seg_file.name, "w") as f:
            f.write("\n".join(lines_out))


# Generamos labels de detección (bbox) a partir de los labels de segmentación,
# para cada partición train/val/test
for split_name in ["train", "val", "test"]:
    seg_dir = SPLIT_ROOT / "labels" / split_name
    det_dir = SPLIT_ROOT / "labels_det" / split_name
    build_detection_labels(seg_dir, det_dir)
    print(f"Labels de detección generados para '{split_name}': "
          f"{len(list(det_dir.glob('*.txt')))} archivos")


Labels de detección generados para 'train': 87 archivos
Labels de detección generados para 'val': 19 archivos
Labels de detección generados para 'test': 20 archivos


## 5. Archivos de configuración del dataset

YOLO necesita un archivo `.yaml` que indique dónde están las imágenes de cada partición y el listado de clases. Creamos **dos** archivos: uno para detección (apunta a `labels_det/`) y otro para segmentación (apunta a `labels/`, que ya contiene los polígonos).


In [7]:
import yaml

names_dict = {i: name for i, name in COCO_CLASSES.items()}

def write_yaml(path, labels_subdir):
    cfg = {
        "path": str(SPLIT_ROOT.resolve()),
        "train": f"images/train",
        "val": f"images/val",
        "test": f"images/test",
        "names": names_dict,
    }
    with open(path, "w") as f:
        yaml.dump(cfg, f, sort_keys=False, allow_unicode=True)

# NOTA: Ultralytics infiere la carpeta de labels reemplazando "images" por "labels"
# en la ruta. Como tenemos dos variantes de labels (labels/ y labels_det/), creamos
# una copia de la estructura de imágenes apuntando a la carpeta de labels correcta
# mediante symlinks, para no duplicar las imágenes en disco.

def make_task_root(task_labels_dirname, task_root_name):
    task_root = Path(f"./dataset_coco/{task_root_name}")
    for split_name in ["train", "val", "test"]:
        img_src = (SPLIT_ROOT / "images" / split_name).resolve()
        lbl_src = (SPLIT_ROOT / task_labels_dirname / split_name).resolve()
        img_dst = task_root / "images" / split_name
        lbl_dst = task_root / "labels" / split_name
        img_dst.parent.mkdir(parents=True, exist_ok=True)
        lbl_dst.parent.mkdir(parents=True, exist_ok=True)
        if img_dst.exists() or img_dst.is_symlink():
            img_dst.unlink()
        if lbl_dst.exists() or lbl_dst.is_symlink():
            lbl_dst.unlink()
        img_dst.symlink_to(img_src)
        lbl_dst.symlink_to(lbl_src)
    return task_root

DET_ROOT = make_task_root("labels_det", "det_task")
SEG_ROOT = make_task_root("labels", "seg_task")

def write_task_yaml(path, task_root):
    cfg = {
        "path": str(task_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": names_dict,
    }
    with open(path, "w") as f:
        yaml.dump(cfg, f, sort_keys=False, allow_unicode=True)

write_task_yaml("coco_det.yaml", DET_ROOT)
write_task_yaml("coco_seg.yaml", SEG_ROOT)

print("coco_det.yaml y coco_seg.yaml generados.")


coco_det.yaml y coco_seg.yaml generados.


## 6. Modelo de detección YOLO preentrenado

Cargamos `yolov8s.pt`, un modelo YOLOv8 **small** ya preentrenado en el dataset COCO completo (80 clases). Elegimos la variante *small* como punto medio entre velocidad y precisión, apropiado para una GPU de 8GB (RTX 4060).


In [8]:
from ultralytics import YOLO

model_det = YOLO("yolov8s.pt")  # pesos preentrenados en COCO (transfer learning)
print(model_det.info())


YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs
(129, 11166560, 0, 28.816844800000002)


## 7. Entrenamiento del modelo de detección (transfer learning)

En lugar de entrenar desde cero, partimos de los pesos preentrenados (`yolov8s.pt`) y hacemos *fine-tuning* sobre nuestra partición de `coco_det.yaml`. Al arrancar desde pesos ya entrenados en COCO, el modelo converge en pocas épocas.

Parámetros elegidos para una RTX 4060 (8GB VRAM):
- `imgsz=640` (resolución estándar de YOLO)
- `batch=16` (cabe cómodamente en 8GB con yolov8s a 640px; bajar a 8 si hay error de memoria)
- `epochs=30` (suficiente para fine-tuning sobre un subconjunto pequeño; subir si se usa un dataset más grande)


In [9]:
results_det = model_det.train(
    data="coco_det.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project="runs_lab",
    name="coco_det_transfer",
    pretrained=True,   # asegura que se use transfer learning, no entrenamiento desde cero
    exist_ok=True,
)


Ultralytics 8.4.106 🚀 Python-3.11.15 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_det_transfer, nbs=

## 8. Modelo de segmentación YOLO preentrenado

Análogamente, cargamos `yolov8s-seg.pt`, la variante de YOLOv8 preentrenada para **segmentación de instancias** sobre COCO.


In [10]:
model_seg = YOLO("yolov8s-seg.pt")  # pesos preentrenados en COCO (transfer learning)
print(model_seg.info())


YOLOv8s-seg summary: 151 layers, 11,821,056 parameters, 0 gradients, 40.3 GFLOPs
(151, 11821056, 0, 40.3440128)


## 9. Entrenamiento del modelo de segmentación (transfer learning)

Usamos la misma partición 70/15/15 (mismas imágenes) pero apuntando al archivo `coco_seg.yaml`, cuyas etiquetas contienen los polígonos de segmentación.


In [11]:
results_seg = model_seg.train(
    data="coco_seg.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project="runs_lab",
    name="coco_seg_transfer",
    pretrained=True,
    exist_ok=True,
)


Ultralytics 8.4.106 🚀 Python-3.11.15 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_seg.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_seg_transfer, 

## 10. Evaluación del modelo de detección (sobre el conjunto de test)

### Ecuaciones de las métricas

**Precision** — de todo lo que el modelo predijo como positivo, ¿qué fracción era correcta?
$$Precision = \frac{TP}{TP + FP}$$

**Recall** — de todos los objetos reales, ¿qué fracción detectó el modelo?
$$Recall = \frac{TP}{TP + FN}$$

**F1-score** — media armónica entre precision y recall:
$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

**Intersection over Union (IoU)** — mide qué tanto se traslapan la caja/máscara predicha ($A$) y la real ($B$):
$$IoU = \frac{|A \cap B|}{|A \cup B|}$$
Una predicción se cuenta como *True Positive* (TP) si su IoU respecto al ground-truth supera un umbral (por ejemplo 0.50).

**mAP@0.50** — mAP (mean Average Precision) calculado usando un único umbral de IoU = 0.50 para decidir aciertos, promediado sobre todas las clases.

**mAP@0.50:0.95** — el mismo cálculo pero promediado sobre 10 umbrales de IoU (0.50, 0.55, ..., 0.95), lo que da una medida más estricta y completa de la calidad de localización.

Donde $TP$ = verdaderos positivos, $FP$ = falsos positivos, $FN$ = falsos negativos.


In [12]:
metrics_det = model_det.val(data="coco_det.yaml", split="test", device=DEVICE)

precision_det = metrics_det.box.mp        # Precision promedio (todas las clases)
recall_det = metrics_det.box.mr           # Recall promedio
map50_det = metrics_det.box.map50         # mAP@0.50
map5095_det = metrics_det.box.map         # mAP@0.50:0.95
f1_det = (2 * precision_det * recall_det / (precision_det + recall_det)
          if (precision_det + recall_det) > 0 else 0.0)

print("=== Métricas de DETECCIÓN (test set) ===")
print(f"Precision      : {precision_det:.4f}")
print(f"Recall         : {recall_det:.4f}")
print(f"F1-score       : {f1_det:.4f}")
print(f"mAP@0.50       : {map50_det:.4f}")
print(f"mAP@0.50:0.95  : {map5095_det:.4f}")


Ultralytics 8.4.106 🚀 Python-3.11.15 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
Model summary (fused): 73 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4024.6±1026.6 MB/s, size: 62.2 KB)
val: Scanning /home/xandro/Documentos/Certificacion/TAE_IA_Modulo_4/Segmentación_y_detección/dataset_coco/split/labels/test... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 2.2Kit/s 0.0s
val: New cache created: /home/xandro/Documentos/Certificacion/TAE_IA_Modulo_4/Segmentación_y_detección/dataset_coco/split/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 5.8it/s 0.3s0.6s
                   all         20        106      0.715      0.739      0.847      0.683
                person          9         27      0.961      0.667      0.913       0.66
               bicycle          1          2      0.916        0.5      0.8

## 11. Evaluación del modelo de segmentación (sobre el conjunto de test)

### Coeficiente Dice

El coeficiente **Dice** (también llamado F1 a nivel de píxel) mide la similitud entre la máscara predicha ($A$) y la máscara real ($B$):

$$Dice = \frac{2 \, |A \cap B|}{|A| + |B|} = \frac{2\,TP}{2\,TP + FP + FN}$$

Se relaciona con IoU mediante: $Dice = \dfrac{2 \cdot IoU}{1 + IoU}$

Las demás métricas (Mask Precision, Mask Recall, Mask mAP@0.50, Mask mAP@0.50:0.95, IoU) se calculan igual que en detección, pero comparando **máscaras a nivel de píxel** en vez de bounding boxes.


In [13]:
metrics_seg = model_seg.val(data="coco_seg.yaml", split="test", device=DEVICE)

mask_precision = metrics_seg.seg.mp
mask_recall = metrics_seg.seg.mr
mask_map50 = metrics_seg.seg.map50
mask_map5095 = metrics_seg.seg.map

print("=== Métricas de SEGMENTACIÓN — máscaras (test set) ===")
print(f"Mask Precision     : {mask_precision:.4f}")
print(f"Mask Recall        : {mask_recall:.4f}")
print(f"Mask mAP@0.50      : {mask_map50:.4f}")
print(f"Mask mAP@0.50:0.95 : {mask_map5095:.4f}")


Ultralytics 8.4.106 🚀 Python-3.11.15 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,810,560 parameters, 0 gradients, 40.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3965.0±2043.9 MB/s, size: 67.4 KB)
val: Scanning /home/xandro/Documentos/Certificacion/TAE_IA_Modulo_4/Segmentación_y_detección/dataset_coco/split/labels/test.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 8.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.7it/s 0.4s1.1s
                   all         20        106      0.745      0.764      0.849      0.697      0.745      0.764      0.844      0.601
                person          9         27          1      0.684      0.897      0.621          1      0.684      0.891      0.467
               bicycle          1          2      0.967          1     

In [15]:
# IoU y Dice promedio calculados directamente sobre las predicciones del test set,
# comparando cada máscara predicha contra su máscara ground-truth correspondiente.
import numpy as np
import cv2

def mask_iou_dice(pred_mask: np.ndarray, gt_mask: np.ndarray):
    """pred_mask y gt_mask son arreglos binarios (0/1) del mismo tamaño."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    iou = intersection / union if union > 0 else 0.0
    dice = (2 * intersection) / (pred_mask.sum() + gt_mask.sum()) \
        if (pred_mask.sum() + gt_mask.sum()) > 0 else 0.0
    return iou, dice


test_images_seg = sorted((SEG_ROOT / "images" / "test").glob("*.jpg"))
iou_scores, dice_scores = [], []

for img_path in test_images_seg:
    result = model_seg.predict(source=str(img_path), device=DEVICE, verbose=False)[0]
    label_path = SEG_ROOT / "labels" / "test" / (img_path.stem + ".txt")
    if result.masks is None or not label_path.exists():
        continue

    h, w = result.orig_shape
    # Máscara ground-truth: reconstruimos el polígono normalizado del archivo .txt
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            coords = list(map(float, parts[1:]))
            pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
            pts[:, 0] *= w
            pts[:, 1] *= h
            cv2.fillPoly(gt_mask, [pts.astype(np.int32)], 1)

    # Combinamos todas las máscaras predichas en una sola máscara binaria
    pred_mask = result.masks.data.cpu().numpy().sum(axis=0)
    pred_mask = (pred_mask > 0).astype(np.uint8)
    pred_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)

    iou, dice = mask_iou_dice(pred_mask, gt_mask)
    iou_scores.append(iou)
    dice_scores.append(dice)

mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
mean_dice = float(np.mean(dice_scores)) if dice_scores else 0.0

print(f"IoU promedio (test)  : {mean_iou:.4f}")
print(f"Dice promedio (test) : {mean_dice:.4f}")


IoU promedio (test)  : 0.8089
Dice promedio (test) : 0.8892


### Resumen de todas las métricas requeridas

In [16]:
import pandas as pd

summary = pd.DataFrame({
    "Métrica": [
        "Precision (det)", "Recall (det)", "F1-score (det)",
        "mAP@0.50 (det)", "mAP@0.50:0.95 (det)",
        "Mask Precision", "Mask Recall",
        "Mask mAP@0.50", "Mask mAP@0.50:0.95",
        "IoU", "Dice",
    ],
    "Valor": [
        precision_det, recall_det, f1_det, map50_det, map5095_det,
        mask_precision, mask_recall, mask_map50, mask_map5095,
        mean_iou, mean_dice,
    ],
})
summary["Valor"] = summary["Valor"].round(4)
summary


,Métrica,Valor
0,Precision (det),0.7147
1,Recall (det),0.7394
2,F1-score (det),0.7268
3,mAP@0.50 (det),0.8467
4,mAP@0.50:0.95 (det),0.6833
5,Mask Precision,0.7447
6,Mask Recall,0.7637
7,Mask mAP@0.50,0.8438
8,Mask mAP@0.50:0.95,0.6015
9,IoU,0.8089


## 12. Visualización de resultados

Para varias imágenes del conjunto de test mostramos, en una sola figura:

1. Imagen original con anotaciones **ground-truth** (caja + máscara real)
2. Predicción del modelo de **detección** (cajas + score de confianza)
3. Predicción del modelo de **segmentación** (máscaras + score de confianza)


In [17]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_gt(ax, img_path, seg_root):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    label_path = seg_root / "labels" / "test" / (img_path.stem + ".txt")
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                coords = list(map(float, parts[1:]))
                pts = np.array(coords).reshape(-1, 2)
                pts[:, 0] *= w
                pts[:, 1] *= h
                poly = patches.Polygon(pts, closed=True, fill=False,
                                        edgecolor="lime", linewidth=2)
                ax.add_patch(poly)
                ax.text(pts[:, 0].min(), pts[:, 1].min() - 4,
                        COCO_CLASSES[cls_id], color="lime", fontsize=8,
                        backgroundcolor="black")
    ax.set_title("Ground truth")
    ax.axis("off")


def draw_detection(ax, img_path):
    result = model_det.predict(source=str(img_path), device=DEVICE, verbose=False)[0]
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  fill=False, edgecolor="red", linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{COCO_CLASSES[cls_id]} {conf:.2f}",
                color="red", fontsize=8, backgroundcolor="black")
    ax.set_title("Detección (predicción)")
    ax.axis("off")


def draw_segmentation(ax, img_path):
    result = model_seg.predict(source=str(img_path), device=DEVICE, verbose=False)[0]
    ax.imshow(result.plot(boxes=False)[:, :, ::-1])  # plot() dibuja máscaras + conf
    ax.set_title("Segmentación (predicción)")
    ax.axis("off")


N_EXAMPLES = 4
sample_imgs = test_images_seg[:N_EXAMPLES]

fig, axes = plt.subplots(N_EXAMPLES, 3, figsize=(15, 5 * N_EXAMPLES))
for i, img_path in enumerate(sample_imgs):
    draw_gt(axes[i, 0], img_path, SEG_ROOT)
    draw_detection(axes[i, 1], img_path)
    draw_segmentation(axes[i, 2], img_path)

plt.tight_layout()
plt.savefig("resultados_visualizacion.png", dpi=150, bbox_inches="tight")
plt.show()


<Figure size 1500x2000 with 12 Axes>

## 13. Conclusiones

### Resultados obtenidos (conjunto de test)

| Métrica | Detección | Segmentación |
|---|---|---|
| Precision | 0.7147 | 0.7447 (mask) |
| Recall | 0.7394 | 0.7637 (mask) |
| F1-score | 0.7268 | — |
| mAP@0.50 | 0.8467 | 0.8438 (mask) |
| mAP@0.50:0.95 | 0.6833 | 0.6015 (mask) |
| IoU | — | 0.8089 |
| Dice | — | 0.8892 |

### Sobre el transfer learning

Partir de pesos preentrenados en COCO (`yolov8s.pt` / `yolov8s-seg.pt`) resultó muy efectivo incluso con un conjunto de entrenamiento pequeño (89 imágenes): ambos modelos alcanzaron un mAP@0.50 superior a 0.84 en test. Esto confirma la ventaja del transfer learning frente a entrenar desde cero: el modelo ya conocía las 80 clases de COCO y solo necesitó ajustar sus pesos al subconjunto particular, en lugar de aprender representaciones visuales desde el inicio.

Se observó que la mejor época en varios entrenamientos ocurrió muy temprano (época 1-2), con ligeras fluctuaciones a la baja después. Esto es coherente con un conjunto de validación pequeño (19 imágenes): las métricas son sensibles al ruido estadístico de tener pocos ejemplos, no señal de que el modelo empeore de forma real. Ultralytics gestiona esto automáticamente guardando el mejor checkpoint (`best.pt`) según el *fitness* de cada época.

### IoU/Dice vs. mAP: una diferencia metodológica relevante

Es interesante notar que el IoU (0.81) y el Dice (0.89) resultaron notablemente más altos que el Mask mAP@0.50:0.95 (0.60). Esto se debe a que se midieron de formas distintas:

- **IoU/Dice** se calcularon comparando la unión de *todas* las máscaras predichas contra la unión de *todas* las máscaras reales de cada imagen — una medida a nivel de escena, que "perdona" errores de asociación entre instancias individuales.
- **Mask mAP** exige emparejar correctamente cada instancia individual con su clase y su score de confianza, siendo una medida mucho más estricta.

Esta diferencia ilustra que ambas métricas responden preguntas distintas: IoU/Dice dicen "qué tan bien se cubre el área de los objetos en general", mientras que mAP dice "qué tan bien se detecta y clasifica cada instancia por separado".

### Clases con bajo desempeño

Las clases `fork`, `dining table`, `sink` y `handbag` obtuvieron los mAP más bajos del conjunto (por debajo de 0.35 en varios casos). En todas ellas, el subconjunto `coco128-seg` contenía muy pocas instancias (1-3 objetos), insuficientes para que el modelo generalizara bien durante el fine-tuning. Esto sugiere que el desempeño por clase está fuertemente correlacionado con la cantidad de ejemplos disponibles, más que con la dificultad intrínseca del objeto.

### Observaciones cualitativas (visualización)

La inspección visual de las predicciones reveló patrones consistentes con las métricas:

- **Casos exitosos:** en imágenes con objetos bien representados en el dataset (cebra, persona con sombrilla), tanto las cajas de detección como las máscaras de segmentación siguieron fielmente los contornos reales, incluso separando correctamente objetos superpuestos.
- **Falso positivo:** en la imagen de la motocicleta, el modelo generó una detección espuria de "person" con confianza muy baja (~0.12) sobre una zona sin ninguna persona real — probablemente por textura/forma ambigua, un error típico de modelos entrenados con pocos datos.
- **Falso negativo:** en la escena del skatepark (múltiples personas con oclusión parcial y poses poco comunes), el modelo omitió por completo una de las tres personas presentes en el ground-truth. Este tipo de escena — con múltiples instancias de la misma clase en poses no estándar — es precisamente donde más se nota la limitación de entrenar con un subconjunto de 128 imágenes en vez del dataset COCO completo.

### Limitaciones y trabajo futuro

- El tamaño reducido del dataset (`coco128-seg`) limita la capacidad del modelo de generalizar a clases con pocas instancias y a escenas con múltiples objetos superpuestos.
- Escalar el entrenamiento a un subconjunto mediano (2,000-5,000 imágenes) o al dataset COCO completo debería mejorar notablemente el mAP@0.50:0.95 y reducir los falsos negativos observados en escenas complejas.
- Podría explorarse el uso de más épocas combinado con *early stopping* basado en `val` para evitar el ligero sobreajuste observado tras las primeras épocas.

*Entrenamiento realizado en GPU local: NVIDIA GeForce RTX 4060 Laptop (8GB VRAM). Tiempo total de entrenamiento: ~47s (detección) + ~68s (segmentación) para 30 épocas cada uno.*